# Exercise 02 — Guide: AlphaFold prediction and confidence

## Learning Objectives

Take a real, extensively studied protein, pull **AlphaFold's actual prediction for it from
the public API**, and work out how much of it to believe. You will:

- fetch a prediction and **know exactly where every number came from**
- read **pLDDT** — and see why the mean hides more than it reveals
- read **PAE**, which answers a question pLDDT cannot
- read **AlphaMissense**, a metric about *function* rather than geometry
- compare the prediction against a **crystal structure** and measure the difference
- find what the model leaves out entirely

**No GPU required.** Everything here is HTTP calls and plotting — it runs on a plain CPU
runtime in about a minute.

## About AI and This Exercise

**What AI CAN help with:** understanding AlphaFold's metrics, syntax, and general
protein-folding principles.

**What YOU must demonstrate:** hypothesis formulation (predictions before running
anything), critical evaluation of what the numbers actually support, and biological
reasoning connecting sequence to structure to function.

**Exercise 02 is two notebooks:**

| Notebook | What it is |
|---|---|
| **`ex02_guide.ipynb`** (this one) | The full worked analysis on human p53 / PDB **3D08**. Every cell is filled in and explained. Nothing here is graded. |
| **`ex02_workbook.ipynb`** | The graded notebook. Open it once you have finished here. |

# Alphafold

[AlphaFold](https://deepmind.google/technologies/alphafold/), an AI system developed by DeepMind, has solved the complex protein-folding problem, allowing for almost instant and highly accurate predictions of protein structures, which are crucial for understanding cellular functions and advancing medical research. Recognized by the [Critical Assessment of protein Structure Prediction community](https://www.predictioncenter.org/), AlphaFold has [significantly](https://www.predictioncenter.org/casp14/zscores_final.cgi) expanded the availability of protein structure data through the freely accessible AlphaFold Protein Structure Database.

---

## The protein: p53 and the 3D08 crystal

We take a real, extensively studied protein, pull **AlphaFold's actual prediction for it from the public API**, and
work out how much of it to believe.

The protein is the **core (DNA-binding) domain of human p53** — "the guardian of the
genome", the most commonly mutated gene in human cancer. PDB entry **3D08** carries a real
cancer-hotspot mutation, **R249S**, and coordinates a structural Zn²⁺ ion required for the
domain to fold.

We will not re-run AlphaFold here. The **AlphaFold Protein Structure Database** already
holds a prediction for every UniProt sequence, and exposes it through a REST API — so we
can fetch the model, its per-residue confidence, and its PAE matrix directly, then plot and
interrogate all three.

**How you'd reason about this before running anything (a worked example of
predict-first):** p53 (UniProt `P04637`) is 393 residues, but 3D08's crystal resolved only
residues 97-287 — the rest was never modelled. Before fetching a single number, the
biologically motivated prediction is: confidence will be **split**, not uniform, and the
*confident* part will be the crystallised core domain. A domain that crystallises cleanly
has a single well-defined fold, which is exactly what a predictor has abundant evidence
for. The regions a crystal never resolved are the suspicious ones.

**This is the same evidence you already know how to gather.** In ex01 you read two
crystallographic signals off a structure: which residues the crystal **failed to resolve**,
and which resolved residues carry the **highest B-factors**. Both say the same thing — that
region is mobile — and both are the natural input to a guess about where a predictor will
be least confident. Watch what the real numbers do with that prediction.

In [ ]:
# Setup. Everything in this notebook runs on a plain CPU runtime.
try:
    from google.colab import drive

    is_google_colab = True
except ImportError:
    is_google_colab = False

if is_google_colab:
    %pip install numpy==2.1.3 pandas==2.2.3 matplotlib==3.10.0 py3dmol==2.4.0

import warnings

warnings.filterwarnings("ignore")

import io
import json
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3Dmol
import requests

print("Libraries loaded -- no GPU needed")


## Where this data comes from

Before plotting anything, it is worth being precise about **what we are downloading and
from where**. "The model says 95" means nothing if you cannot say which model, which
version, and which file the number came from.

Everything below comes from the **AlphaFold Protein Structure Database**, and the whole
chain starts from a single UniProt accession:

```
UniProt accession  ->  https://alphafold.ebi.ac.uk/api/prediction/{accession}
                            |
                            +-- globalMetricValue      the headline mean pLDDT
                            +-- fractionPlddt*         how much sits in each confidence band
                            +-- pdbUrl        -------> the predicted structure   (.pdb)
                            +-- plddtDocUrl   -------> per-residue confidence    (.json)
                            +-- paeDocUrl     -------> predicted aligned error   (.json)
                            +-- amAnnotationsUrl ----> every scored substitution (.csv)
                            +-- msaUrl        -------> the alignment used        (.a3m)
```

One call returns a small JSON record that **contains the URLs of everything else**. You
never construct those URLs yourself, which is why this keeps working when AlphaFold DB
bumps its version suffix (`_v4`, `_v6`, ...).

For p53 the accession is **`P04637`**. Note what that means: AlphaFold DB is keyed by
**UniProt sequence, not by PDB entry**. There is no prediction "for 3D08". There is a
prediction for the canonical human p53 sequence, which we then compare against the 3D08
crystal. That distinction matters, because 3D08 is the R249S mutant and the AlphaFold
model is wild-type.


### The AlphaFold DB helpers

Five functions do all the talking to the database in this notebook, and everything below
calls them rather than repeating their contents. These are the ones you will need again
on your own protein.

| Helper | Gives you |
|---|---|
| `prediction_record(uniprot)` | the JSON record above, as one row |
| `download_prediction_files(entry, directory)` | the model, confidence and PAE files, on disk |
| `per_residue_confidence(entry)` | pLDDT and its band, indexed by residue number |
| `pae_matrix(entry)` | the n x n predicted aligned error, indexed by residue number |
| `confidence_bands(entry)` | the four `fractionPlddt*` fractions |


In [ ]:
def prediction_record(uniprot, fragment=1):
    """AlphaFold DB's record for one UniProt accession, as a single row.

    The record carries the summary metrics and the URLs of every other file, so nothing
    below ever builds an AlphaFold DB URL by hand. Very long proteins come back as more
    than one fragment; most, including p53, fit in `F1`.
    """
    entries = pd.json_normalize(
        requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot}").json()
    )
    return entries.set_index("modelEntityId").loc[f"AF-{uniprot}-F{fragment}"]


def download_prediction_files(entry, directory):
    """Save the three files the record points at. Returns {name: path}.

    A URL the database does not serve is reported and skipped rather than stopping the
    download of the others.
    """
    directory.mkdir(exist_ok=True)
    wanted = {"model.pdb": "pdbUrl", "confidence.json": "plddtDocUrl", "pae.json": "paeDocUrl"}
    saved = {}
    for name, field in wanted.items():
        response = requests.get(entry[field])
        target = directory / f"{entry.name}-{name}"
        if response.status_code != 200:
            print(f"    {name:<18} HTTP {response.status_code}  <-- unavailable, skipped")
            continue
        target.write_bytes(response.content)
        saved[name] = target
        print(f"    {name:<18} {len(response.content):>9,} bytes  -> {target}")
    return saved


def per_residue_confidence(entry):
    """pLDDT and its confidence category, indexed by residue number.

    The file is three parallel arrays, which is already a table. Indexing by residue
    number means slicing a range later reads like the biology.
    """
    return (
        pd.read_json(entry["plddtDocUrl"])
        .rename(columns={"confidenceScore": "pLDDT", "confidenceCategory": "category"})
        .set_index("residueNumber")
    )


def pae_matrix(entry):
    """The n x n predicted aligned error, indexed by residue number on both axes."""
    record = pd.read_json(entry["paeDocUrl"]).iloc[0]
    pae = pd.DataFrame(record["predicted_aligned_error"])
    pae.index += 1              # residue numbering starts at 1, not 0
    pae.columns += 1
    return pae


def confidence_bands(entry):
    """The four fractionPlddt* bands, as a Series labelled by their pLDDT range."""
    labels = {
        "fractionPlddtVeryLow": "Very low (<50)",
        "fractionPlddtLow": "Low (50-70)",
        "fractionPlddtConfident": "Confident (70-90)",
        "fractionPlddtVeryHigh": "Very high (>90)",
    }
    return pd.Series({label: entry[field] for field, label in labels.items()})


In [ ]:
UNIPROT = "P04637"          # human p53
DATA = Path("afdb_data")
DATA.mkdir(exist_ok=True)

entry = prediction_record(UNIPROT)
(DATA / f"afdb_{UNIPROT}.json").write_text(entry.to_json(indent=2))

print(f"Model: {entry.name}  (pipeline: {entry['toolUsed']})")
print(f"Sequence length: {entry['sequenceEnd'] - entry['sequenceStart'] + 1} residues")
print(f"Global mean pLDDT: {entry['globalMetricValue']:.1f}\n")

print("  fields we will actually use:")
for key in ("uniprotSequence", "globalMetricValue", "modelCreatedDate", "latestVersion",
            "pdbUrl", "plddtDocUrl", "paeDocUrl", "amAnnotationsUrl", "msaUrl"):
    if key in entry:
        value = entry[key]
        shown = f"{value[:52]}..." if isinstance(value, str) and len(value) > 55 else value
        print(f"    {key:<20} {shown}")

print("\n  downloading:")
files = download_prediction_files(entry, DATA)

# msaUrl is listed above but has been returning HTTP 403 since 2026-08; we do not rely on it.
msa_status = requests.get(entry["msaUrl"]).status_code
print(f"    {'msa.a3m':<18} HTTP {msa_status}" + ("  <-- not public; alignment work lives in ex01"
                                                  if msa_status != 200 else ""))


### What each file looks like

Knowing the *shape* of these files is what lets you use them. Run the next cell and read
the output alongside this table:

| File | Structure |
|---|---|
| `afdb_P04637.json` | one flat record — summary metrics plus the URLs above |
| `...confidence.json` | three parallel lists: `residueNumber`, `confidenceScore`, `confidenceCategory` |
| `...pae.json` | a list with one object holding `predicted_aligned_error` (an *n × n* matrix) and `max_predicted_aligned_error` |
| `...model.pdb` | a normal PDB file — **with pLDDT written into the B-factor column** |

That last one is the detail people miss. AlphaFold does not ship a separate confidence
file for the structure; it **reuses the B-factor column**. So any tool that colours a
structure by B-factor — PyMOL, ChimeraX, py3Dmol — colours an AlphaFold model by
confidence for free. It also means you must **never** read those numbers as real
crystallographic B-factors. Same column, completely different meaning.

In [ ]:
# Read each file back from disk and show its actual shape. Everything after this cell
# goes through the helpers instead; this is the one look at the raw files.
conf = json.loads(files["confidence.json"].read_text())
print("confidence.json")
print(f"  keys        : {list(conf.keys())}")
print(f"  n residues  : {len(conf['residueNumber'])}")
print(f"  first 5     : residue {conf['residueNumber'][:5]}")
print(f"                score   {conf['confidenceScore'][:5]}")
print(f"                category{conf['confidenceCategory'][:5]}   (D/L/M/H = disordered->high)")

pae_obj = json.loads(files["pae.json"].read_text())[0]
print("\npae.json")
print(f"  keys        : {list(pae_obj.keys())}")
print(f"  matrix      : {len(pae_obj['predicted_aligned_error'])} x "
      f"{len(pae_obj['predicted_aligned_error'][0])}")
print(f"  max_predicted_aligned_error = {pae_obj['max_predicted_aligned_error']}")

atom_lines = [l for l in files["model.pdb"].read_text().splitlines() if l.startswith("ATOM")]
print("\nmodel.pdb")
print(f"  ATOM records: {len(atom_lines):,}")
print(f"  first ATOM  : {atom_lines[0][:66]}")
print(f"                                                     ^^^^^^ B-factor column = pLDDT")

# Prove the B-factor column really is the pLDDT from confidence.json
first_ca = next(l for l in atom_lines if l[12:16].strip() == "CA")
print(f"\n  residue 1 pLDDT from confidence.json : {conf['confidenceScore'][0]}")
print(f"  residue 1 B-factor from model.pdb    : {float(first_ca[60:66])}")


### pLDDT along the sequence

The global mean above is one number for all 393 residues. Before interpreting it, zoom in
on exactly the region 3D08's crystal actually covers (residues 97-287).

| Helper | Gives you |
|---|---|
| `plot_plddt(confidence, highlight=..., ax=...)` | the per-residue confidence trace, with a range shaded |
| `plot_pae(pae, ...)` | the PAE matrix as a heatmap |

Both take an `ax`, so the same trace can be drawn on its own or as one panel of a larger
figure further down.


In [ ]:
def plot_plddt(confidence, highlight=None, title=None, ax=None):
    """pLDDT along the sequence. `highlight` shades a (start, end) residue range."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 4))
    ax.plot(confidence.index, confidence["pLDDT"], color="tab:blue", lw=1)
    ax.axhline(70, color="grey", ls="--", lw=1, label="pLDDT = 70 (confident cutoff)")
    if highlight is not None:
        ax.axvspan(highlight[0], highlight[1], color="tab:green", alpha=0.15,
                   label=f"resolved range ({highlight[0]}-{highlight[1]})")
    ax.set_xlabel("Residue number")
    ax.set_ylabel("pLDDT")
    ax.set_ylim(0, 100)
    ax.set_title(title or "AlphaFold DB per-residue confidence")
    ax.legend(loc="lower center")
    return ax


def plot_pae(pae, title="Predicted Aligned Error (PAE)", ax=None):
    """The PAE matrix as a heatmap, dark = confident.

    Takes the DataFrame from `pae_matrix` or a plain array: the AlphaFold Server writes
    one of those, and the same picture has to be readable for both.
    """
    values = np.asarray(pae)
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(values, cmap="Greens_r", vmin=0, vmax=values.max(),
                   extent=[1, values.shape[1], values.shape[0], 1])
    ax.set_xlabel("Aligned residue")
    ax.set_ylabel("Scored residue")
    ax.set_title(title)
    ax.figure.colorbar(im, ax=ax, label="Expected error (A)")
    return ax


In [ ]:
confidence = per_residue_confidence(entry)

CRYSTAL_START, CRYSTAL_END = 97, 287        # 3D08's own resolved range
core = confidence.loc[CRYSTAL_START:CRYSTAL_END, "pLDDT"]

print(f"AlphaFold mean pLDDT over 3D08's resolved range "
      f"({CRYSTAL_START}-{CRYSTAL_END}): {core.mean():.1f}")
print(f"Fraction of that range with pLDDT < 70: {(core < 70).mean()*100:.1f}%")

print("\npLDDT right around the R249S hotspot position (240-260):")
print(confidence.loc[240:260].to_string())


In [ ]:
plot_plddt(confidence, highlight=(CRYSTAL_START, CRYSTAL_END),
           title=f"AlphaFold DB per-residue confidence for human p53 ({UNIPROT})")
plt.show()


### Predicted Aligned Error (PAE)

pLDDT tells you how confident AlphaFold is about *where a residue sits locally*. It does
not tell you how confident AlphaFold is about the **relative position of two residues
that are far apart in sequence** — two domains can each individually have high pLDDT
while AlphaFold is still unsure how they're oriented relative to each other. That's what
**PAE** (Predicted Aligned Error) is for: a full residue-by-residue matrix, where a low
value at (i, j) means AlphaFold is confident about residue j's position *if you fix
residue i in place*. AlphaFold DB publishes the full PAE matrix for every entry, the
same way it publishes pLDDT.

In [ ]:
pae = pae_matrix(entry)
print(f"PAE matrix: {pae.shape[0]} x {pae.shape[1]} residues")
print(f"Largest expected error in this entry: {pae.values.max():.1f} A")

plot_pae(pae, title="Predicted Aligned Error (PAE), full-length p53")
plt.show()

# Because it is a DataFrame indexed by residue, questions about regions are one line each:
core_block = pae.loc[CRYSTAL_START:CRYSTAL_END, CRYSTAL_START:CRYSTAL_END]
print(f"\nMean PAE within the core domain  : {core_block.values.mean():.1f} A")
print(f"Mean PAE core vs. N-terminal tail : "
      f"{pae.loc[CRYSTAL_START:CRYSTAL_END, 1:CRYSTAL_START-1].values.mean():.1f} A")


**What just happened:** the *global* average for full-length p53 (~75, with ~40% of
the sequence in AlphaFold DB's low/very-low confidence bins) looks like a real weak
spot. But restricted to exactly the domain 3D08 crystallized (97-287), AlphaFold is
actually **very** confident (mean pLDDT ≈ 95-96, only ~1% below 70) — including right at
the R249 mutation hotspot itself. The pLDDT plot above shows this directly: a sharp,
visible step at the boundaries of the shaded region, not a gradual decline.

The low-confidence 40% comes almost entirely from p53's N- and C-terminal regulatory
regions, which are **intrinsically disordered** — they genuinely have no single fixed
shape in solution. AlphaFold being unconfident there isn't a prediction failure; it's the
correct answer for a region that doesn't fold into one structure.

This is exactly the kind of thing to check before trusting (or dismissing) an AlphaFold
confidence score: a single overall number can hide very different stories in different
parts of the same protein. You now have two independent views of that story — pLDDT and
PAE — rather than one summary statistic.

*(A third view, the depth of the MSA AlphaFold used, is a natural next question. Building
and reading an alignment is exactly what you did in ex01, so that is where the
conservation work lives in this course.)*

---

## Beyond pLDDT: the other metrics AlphaFold DB gives you

pLDDT and PAE are the two everyone quotes, and they answer the same *kind* of question —
**how sure is the model about where things are?** The prediction record carries more than
that, and one of the extras answers a completely different question: **does changing this
residue break the protein?**

### The confidence composition

Before the new metric, one small thing already in the record. Alongside the headline mean,
AlphaFold DB reports what fraction of the protein sits in each confidence band. A single
average hides the shape of the distribution; these four numbers do not.

In [ ]:
bands = confidence_bands(entry)

print(f"Mean pLDDT {entry['globalMetricValue']:.1f} is made of:")
for label, fraction in bands.items():
    print(f"  {label:<20} {fraction*100:5.1f}%  {'#' * int(fraction * 50)}")

usable = bands["Confident (70-90)"] + bands["Very high (>90)"]
print(f"\n  above 70 (usable): {usable*100:.1f}%")
print(f"  below 70         : {(1 - usable)*100:.1f}%")


**Read that composition, not the mean.** p53's average of ~75 sounds mediocre. The
breakdown says something quite different: **over half the protein is above pLDDT 90**, and
almost 30% is below 50. This is not a uniformly mediocre prediction — it is an excellent
prediction of part of the protein and an honest admission of ignorance about the rest. The
mean is the one number that describes neither half.

### AlphaMissense: a metric about function, not geometry

The prediction record also carries `amAnnotationsUrl`, which points at **AlphaMissense** —
a separate DeepMind model that scores **every possible amino-acid substitution** in the
protein for how likely it is to be disease-causing.

This is a different axis entirely from pLDDT:

| | asks | high value means |
|---|---|---|
| **pLDDT** | can I place this atom? | the geometry is confidently predicted |
| **AlphaMissense** | does changing this residue break the protein? | the position is intolerant to substitution |

A residue can be confidently placed *and* freely mutable — a surface loop with a rigid
backbone. It can also be poorly placed *and* critical. The two metrics are not
interchangeable, and putting them side by side is far more informative than either alone.

For a 393-residue protein you get **393 × 19 ≈ 7,500 scored variants**, which is a
heatmap, not a number.

### The AlphaMissense helpers

| Helper | Gives you |
|---|---|
| `alphamissense(entry)` | every scored substitution, with position, wild type and mutant split out |
| `missense_heatmap(am, start, end, mark=...)` | the 20 x N grid over a residue window |

`alphamissense` returns an empty table rather than raising when an entry has no
annotations. Coverage is human proteins, so that is a real outcome, not a bug.


In [ ]:
AA_ORDER = list("AVLIMFWYGPSTCNQHKRDE")

AM_COLUMNS = ["protein_variant", "am_pathogenicity", "am_class", "pos", "wt", "mut"]


def alphamissense(entry):
    """Every scored substitution for this entry, with the variant string split out.

    `pos`, `wt` and `mut` come from parsing `protein_variant` (for example `R249S`), which
    is what makes per-position and per-substitution views one groupby each.
    """
    url = entry.get("amAnnotationsUrl")
    if not isinstance(url, str):
        print("No AlphaMissense annotations for this entry: coverage is human proteins.")
        return pd.DataFrame(columns=AM_COLUMNS)
    am = pd.read_csv(io.StringIO(requests.get(url).text))
    am["pos"] = am["protein_variant"].str.extract(r"(\d+)").astype(int)
    am["wt"] = am["protein_variant"].str[0]
    am["mut"] = am["protein_variant"].str[-1]
    return am


def missense_heatmap(am, start, end, mark=None, ax=None):
    """Every substitution over a residue window, red = likely pathogenic.

    A 20 x N grid is only readable over a window, so pick the stretch you want to argue
    about. `mark` draws a line on one residue.
    """
    window = am[am["pos"].between(start, end)]
    grid = (window.pivot_table(index="mut", columns="pos", values="am_pathogenicity")
            .reindex(AA_ORDER))
    if ax is None:
        _, ax = plt.subplots(figsize=(14, 5))
    im = ax.imshow(grid.values, cmap="RdBu_r", vmin=0, vmax=1, aspect="auto")
    ax.set_yticks(range(len(AA_ORDER)))
    ax.set_yticklabels(AA_ORDER)
    ax.set_xticks(range(0, grid.shape[1], 5))
    ax.set_xticklabels(grid.columns[::5], rotation=90)
    ax.set_xlabel("residue number")
    ax.set_ylabel("substituted to")
    ax.set_title(f"AlphaMissense pathogenicity, residues {start}-{end} "
                 "(red = likely pathogenic)")
    ax.figure.colorbar(im, ax=ax, label="am_pathogenicity")
    if mark is not None and mark in grid.columns:
        ax.axvline(list(grid.columns).index(mark), color="black", lw=1.5, ls="--")
    return ax


In [ ]:
am = alphamissense(entry)

print(f"AlphaMissense: {len(am):,} variants over positions "
      f"{am['pos'].min()}-{am['pos'].max()}")
print(f"  classes   : {dict(am['am_class'].value_counts())}")
print("     LPath = likely pathogenic, LBen = likely benign, Amb = ambiguous")

# per-position mean pathogenicity, as a track we can line up against pLDDT
per_pos = am.groupby("pos")["am_pathogenicity"].mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
plot_plddt(confidence, highlight=(CRYSTAL_START, CRYSTAL_END),
           title="Two independent metrics on the same protein", ax=ax1)
ax2.plot(per_pos.index, per_pos.values, color="tab:red", lw=1)
ax2.axhline(0.5, color="grey", ls="--", lw=1)
ax2.axvspan(CRYSTAL_START, CRYSTAL_END, color="tab:green", alpha=0.15)
ax2.set_xlabel("Residue number")
ax2.set_ylabel("mean AlphaMissense\npathogenicity")
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.show()

core_positions = per_pos.index.to_series().between(CRYSTAL_START, CRYSTAL_END)
print(f"\nMean pathogenicity, core {CRYSTAL_START}-{CRYSTAL_END}: "
      f"{per_pos[core_positions.values].mean():.3f}")
print(f"Mean pathogenicity, the tails            : "
      f"{per_pos[~core_positions.values].mean():.3f}")


**The two tracks have the same shape** — the folded core is both confidently predicted and
intolerant to mutation; the tails are neither. But notice they say this for *unrelated*
reasons. pLDDT is high in the core because a deep MSA constrains the geometry.
AlphaMissense is high there because substitutions in a packed core disrupt function. Two
models, two training objectives, one conclusion: **that domain is where the protein does
its work.**

Agreement between independent methods is worth much more than a strong result from one.
This is the single most useful habit in the whole exercise.

### The variant heatmap, and the residue 3D08 is about

Now stop averaging and look at individual substitutions.

In [ ]:
missense_heatmap(am, 230, 275, mark=249)
plt.show()

print("Every substitution at residue 249, worst first:")
print(am[am["pos"] == 249].sort_values("am_pathogenicity", ascending=False)
      [["protein_variant", "am_pathogenicity", "am_class"]].head(8).to_string(index=False))


**Look at what that says.** 3D08's crystal carries **R249S** — a real hepatocellular
carcinoma hotspot — and AlphaMissense scores it **0.9995, likely pathogenic**. Not just
that substitution: essentially *every* substitution at position 249 is predicted damaging.
The column is solid red. Arginine 249 is not merely conserved; it is a position where the
protein tolerates nothing else.

Now cross-check it against something you compute yourself further down. This notebook
locates the residues coordinating 3D08's structural **Zn²⁺** purely from geometry —
Cys176, His179, Cys238, Cys242. AlphaMissense, which has never seen the crystal structure,
scores those four positions at **mean pathogenicity 0.998–1.000**: near-total intolerance.

**Two completely independent lines of evidence — a distance calculation on a crystal, and
a variant-effect model trained on population genetics — converge on the same four
residues.** That convergence is the finding, and neither method alone would have given it
to you.

---

## Making your own prediction: the AlphaFold Server

Everything above **looked up** a prediction that already existed. AlphaFold DB holds
precomputed models for UniProt sequences — which is wonderful when your protein is in
there, and useless when it is not: a mutant, a fragment, a complex, or a protein with a
ligand bound.

For those you run the prediction yourself, on the **[AlphaFold Server](https://alphafoldserver.com/)**.
It runs **AlphaFold 3**, it is free for academic use, and it needs no installation.

**What AF3 can do that AlphaFold DB cannot:**

| | AlphaFold DB | AlphaFold Server (AF3) |
|---|---|---|
| Input | a UniProt entry that already exists | any sequence you paste |
| Protein complexes | no | yes |
| DNA / RNA | no | yes |
| **Ligands, ions, modifications** | **no** | **yes** — by CCD code or SMILES |
| Cost | instant lookup | **20 jobs per day** |

That ligand support is the important one for this course: your own protein comes with a
ligand, and AF3 will predict the **complex** — protein and ligand together — which is a
fundamentally different thing from docking a ligand into a fixed receptor.

### Submitting 3D08's sequence

Let us reproduce, by hand, the prediction we have been reading from the database.

1. Go to <https://alphafoldserver.com/> and sign in with a Google account.
2. Choose **Protein** as the entity type.
3. Paste p53's core domain — residues 97-287, exactly 3D08's resolved range. The cell
   below prints it for you to copy.
4. *(Optional, and the interesting part.)* Add a second entity: **Ion → `ZN`**. 3D08's
   crystal contains a structural zinc, and AlphaFold DB's protein-only model does not.
   This is something you simply cannot ask AlphaFold DB.
5. Name the job, press **Continue and preview job**, then **Confirm and submit job**.
6. Wait — usually a few minutes — then **download** the result. You get a ZIP.

⚠️ **Mind the daily quota: 20 jobs.** Decide what you are asking before you submit, rather
than resubmitting variations. That constraint is realistic; compute is never free.

In [ ]:
# The exact sequence to paste into the AlphaFold Server. The record already carries the
# full UniProt sequence AlphaFold was given, so there is nothing else to fetch.
core_sequence = entry["uniprotSequence"][CRYSTAL_START - 1:CRYSTAL_END]

print(f"p53 core domain, residues {CRYSTAL_START}-{CRYSTAL_END} "
      f"({len(core_sequence)} aa) -- copy this:\n")
print("\n".join(textwrap.wrap(core_sequence, 60)))


### What comes back, and how to read it

Unzip the download. The files that matter:

| File | What it holds |
|---|---|
| `*_model.cif` | the predicted structure (mmCIF, not PDB) |
| `*_summary_confidences.json` | the headline numbers — `ptm`, `iptm`, `ranking_score`, `fraction_disordered`, `has_clash` |
| `*_confidences.json` | the full arrays — `atom_plddts`, `pae`, `contact_probs`, `token_chain_ids` |
| `ranking_scores.csv` | scores for every model produced, so you can see the spread |

Two things to notice before plotting anything.

**pLDDT is per *atom*, not per residue** (`atom_plddts`). AlphaFold DB gave us one value
per residue; AF3 works at atom resolution because it has to handle ligands and nucleic
acids, which do not have residues in the protein sense. To compare against what we plotted
earlier you must aggregate to residues.

**`ptm` and `iptm` are new.** `ptm` scores the whole predicted structure; **`iptm` scores
the *interfaces* between chains** — how confident the model is that two entities are
positioned correctly *relative to each other*. For a single protein `iptm` is meaningless.
For a protein with a ligand it is the number that matters most, and it is the one you will
use in ex04 when comparing co-folding against docking.

Upload your ZIP to this notebook, then:

In [ ]:
# Read an AlphaFold Server result. Point these at your own unzipped download.
SUMMARY = Path("fold_p53_core_summary_confidences.json")   # <-- your filename
FULL    = Path("fold_p53_core_confidences.json")           # <-- your filename

if not SUMMARY.exists():
    print("No AlphaFold Server output found in this directory.")
    print("Upload and unzip your job download, then set the two paths above.")
else:
    summary = json.loads(SUMMARY.read_text())
    print("Summary confidences")
    for key in ("ranking_score", "ptm", "iptm", "fraction_disordered", "has_clash"):
        if key in summary:
            print(f"  {key:<20} {summary[key]}")

    full = json.loads(FULL.read_text())
    atom_plddt = np.array(full["atom_plddts"])
    server_pae = np.array(full["pae"])
    print(f"\n  atom_plddts : {atom_plddt.shape[0]} atoms, mean {atom_plddt.mean():.1f}")
    print(f"  pae         : {server_pae.shape[0]} x {server_pae.shape[1]} tokens")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax1.plot(atom_plddt, lw=0.6, color="tab:blue")
    ax1.axhline(70, color="grey", ls="--", lw=1)
    ax1.set_xlabel("atom index")
    ax1.set_ylabel("pLDDT")
    ax1.set_ylim(0, 100)
    ax1.set_title("Per-atom confidence")
    # Same helper as the database PAE above: per-atom pLDDT is a different plot, PAE is not.
    plot_pae(server_pae, title="Predicted Aligned Error", ax=ax2)
    ax2.set_xlabel("aligned token")
    ax2.set_ylabel("scored token")
    plt.tight_layout()
    plt.show()


### Reading the two plots

**The pLDDT trace** should look like the AlphaFold DB curve you plotted earlier — high and
flat across the folded core. It will not be identical: AF3 is a different model from the
AF2 pipeline that produced the database entry, you gave it only the core domain rather
than the full 393-residue protein, and there is a random seed involved. **Compare the
shapes, not the decimal places.** If your core sits high and flat, the two models agree
about the thing that matters.

**The PAE matrix** is the more informative plot, and the one people skip. Read it as a
map, not a picture:

- **Dark blocks along the diagonal** = regions confidently placed *relative to themselves*
  — a domain.
- **Dark off-diagonal blocks** = two regions confidently placed *relative to each other* —
  they form a rigid unit.
- **Light off-diagonal regions** = the model has no idea how these two parts sit together,
  even if each is individually confident.

That last case is the one pLDDT alone will never tell you. Two domains can each score
pLDDT 95 while their relative orientation is a guess — high confidence in the parts,
none in the assembly.

**If you added the Zn ion**, look at the PAE rows and columns for that token: is the ion
confidently placed relative to the protein? Then check whether the residues nearest it are
the ones you found coordinating zinc in the crystal further down this notebook —
Cys176, His179, Cys238 and Cys242.

---

## 🔭 FRONTIER → Block C (Molecular Dynamics)

**⏭ Signpost:** You'll fully understand *why* this happens in Lectures 7–9, when we
cover molecular dynamics.

**The question:** AlphaFold gave p53's core domain (97-287) a very high, uniform
confidence score (~95-96 mean pLDDT) — but a pLDDT score is a **static, single-snapshot**
measure. Even within this "confident" domain, will every residue be equally *rigid* in an
actual molecular dynamics simulation?

**💡 Hint:** pLDDT measures how confident AlphaFold is about *where* a residue sits in one
predicted structure — it is not, by itself, a measurement of flexibility in solution. A
region can be confidently placed and still move.

**A worked answer, as a model of the reasoning:** no — high pLDDT and rigidity are
different claims about different things. Confidence says "I know where this sits";
flexibility says "this moves once you put it in water at 310 K." The regions to expect
movement in are the surface loops connecting the β-strands of the DNA-binding surface,
and the residues immediately flanking the crystal's own unresolved gaps (identified
further down) — a loop the crystal couldn't pin down is unlikely to sit still in
simulation either, even when AlphaFold is confident about its average position. The
Zn²⁺-coordinating site, by contrast, should be among the *most* rigid: a four-point
metal coordination physically clamps those residues together.

---

## Correlating the AlphaFold model with the 3D08 crystal: what actually changed?

Confidence metrics are AlphaFold's own self-assessment. The more direct check is to put
the AlphaFold model and the real crystal structure **side by side** and measure the
difference directly.

Two things worth knowing *before* looking at the numbers, so a deviation isn't
misread:

- AlphaFold DB's model is predicted from p53's **canonical (wild-type) UniProt
  sequence** (`entry["uniprotSequence"]`, already fetched above). 3D08's crystal carries
  the **R249S mutation**, so this comparison is partly "AlphaFold's WT prediction vs. a
  mutant crystal," not purely "prediction accuracy vs. experiment." A deviation right at
  residue 249 could reflect either.
- 3D08's crystal has a bound **structural Zn²⁺ ion**, coordinated by four residues
  (verified below). AlphaFold's model is **protein-only**: it does not predict ions,
  ligands, or cofactors at all. That's a categorical omission by design, not something an
  RMSD number will show you.


### The structure-comparison helpers

Six functions, and the same rule as before: defined once here, called everywhere below.

| Helper | Gives you |
|---|---|
| `fetch_pdb_text(pdb_id)` | an RCSB entry, as text |
| `atom_records(pdb_text, record=...)` | ATOM or HETATM lines as a DataFrame of coordinates |
| `ca_coordinates(pdb_text, chain=...)` | `{residue number: xyz}` for one chain |
| `superpose(mobile, reference)` | the rotation, both centroids, and the per-residue deviation |
| `apply_transform(pdb_text, ...)` | a whole structure moved by that rotation |
| `residues_near_point(pdb_text, point, radius)` | residues within a distance of any point |


In [ ]:
def fetch_pdb_text(pdb_id):
    """One RCSB entry, as text."""
    return requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb").text


def atom_records(pdb_text, record="ATOM"):
    """ATOM (or HETATM) lines as a DataFrame: chain, residue, atom name and coordinates.

    Fixed-width column parsing, which is the actual PDB format. Splitting on whitespace
    silently misparses lines where columns run together, for instance at 4-digit residue
    numbers. Alternate conformations beyond the first are dropped.
    """
    rows = [
        (line[21], int(line[22:26]), line[17:20].strip(), line[12:16].strip(),
         float(line[30:38]), float(line[38:46]), float(line[46:54]))
        for line in pdb_text.splitlines()
        if line.startswith(record) and line[16] in (" ", "A")
    ]
    return pd.DataFrame(rows, columns=["chain", "residue_number", "residue_name",
                                       "atom_name", "x", "y", "z"])


def ca_coordinates(pdb_text, chain="A"):
    """CA coordinates keyed by residue number, for one chain."""
    atoms = atom_records(pdb_text)
    ca = atoms[(atoms["atom_name"] == "CA") & (atoms["chain"] == chain)]
    return {int(row.residue_number): (row.x, row.y, row.z) for row in ca.itertuples()}


def superpose(mobile, reference):
    """Kabsch superposition of two {residue number: xyz} maps over the residues they share.

    Returns (rotation, mobile centroid, reference centroid, per-residue deviation). The
    overall RMSD is the root mean square of that deviation, which is worth computing at
    the call site rather than hiding here.

    Matching is by residue NUMBER, not by list position: a crystal with unresolved loops
    is not a gap-free span, and pairing by position would silently misalign everything
    after the first gap.
    """
    common = sorted(set(mobile) & set(reference))
    P = np.array([mobile[r] for r in common])
    Q = np.array([reference[r] for r in common])
    P_centroid, Q_centroid = P.mean(axis=0), Q.mean(axis=0)

    U, S, Vt = np.linalg.svd((P - P_centroid).T @ (Q - Q_centroid))
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    rotation = Vt.T @ np.diag([1, 1, d]) @ U.T   # guards against an improper rotation

    aligned = (rotation @ (P - P_centroid).T).T + Q_centroid
    deviation = pd.Series(np.linalg.norm(aligned - Q, axis=1), index=common,
                          name="deviation (A)")
    deviation.index.name = "residue_number"
    return rotation, P_centroid, Q_centroid, deviation


def apply_transform(pdb_text, rotation, from_centroid, to_centroid):
    """A whole structure moved by a rotation from `superpose`, as PDB text.

    Superposing only the matched CA atoms tells you the number; moving every atom is what
    lets you look at the two structures in one view.
    """
    moved = []
    for line in pdb_text.splitlines():
        if line.startswith(("ATOM", "HETATM")):
            xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            new = (rotation @ (xyz - from_centroid)) + to_centroid
            line = f"{line[:30]}{new[0]:8.3f}{new[1]:8.3f}{new[2]:8.3f}{line[54:]}"
        moved.append(line)
    return "\n".join(moved)


def residues_near_point(pdb_text, point, radius):
    """Residue numbers with any protein atom within `radius` angstroms of a point."""
    atoms = atom_records(pdb_text)
    distance = np.linalg.norm(atoms[["x", "y", "z"]].values - np.asarray(point), axis=1)
    # Plain ints, not numpy ones: py3Dmol JSON-encodes any selection it is handed,
    # and a numpy int64 in there fails to serialise.
    return sorted(int(r) for r in atoms.loc[distance < radius, "residue_number"].unique())


def show_superposition(reference_text, mobile_text, highlight=(), ligand=None,
                       width=900, height=550):
    """Two structures in one view: reference blue, mobile orange, `highlight` as sticks."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(reference_text, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "skyblue"}})
    view.addModel(mobile_text, "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "orange"}})
    if len(highlight):
        view.addStyle({"model": 0, "resi": list(highlight)}, {"stick": {"color": "skyblue"}})
        view.addStyle({"model": 1, "resi": list(highlight)}, {"stick": {"color": "orange"}})
    if ligand:
        view.addStyle({"model": 0, "resn": ligand}, {"sphere": {"color": "grey", "radius": 0.6}})
    view.zoomTo({"model": 0, "resi": list(highlight)} if len(highlight) else {})
    return view


In [ ]:
# AlphaFold's model is already on disk from the download above; the crystal is one call.
af_model_text = files["model.pdb"].read_text()
crystal_text = fetch_pdb_text("3D08")

af_ca = ca_coordinates(af_model_text)
crystal_ca = ca_coordinates(crystal_text)
print(f"AlphaFold model: {len(af_ca)} CA atoms, range {min(af_ca)}-{max(af_ca)}")
print(f"3D08 crystal: {len(crystal_ca)} CA atoms, range {min(crystal_ca)}-{max(crystal_ca)}")

common_residues = sorted(set(af_ca) & set(crystal_ca))
unresolved = [r for r in range(min(common_residues), max(common_residues) + 1)
              if r not in crystal_ca]
print(f"Matched (by residue number) residues: {len(common_residues)}")
print(f"Residues with no crystal coordinates inside that range (unresolved loops): {unresolved}")


In [ ]:
# Superimpose the AlphaFold model onto the crystal (Kabsch), then read the deviation
# residue by residue: the overall number hides where the disagreement actually is.
rotation, af_centroid, crystal_centroid, deviation = superpose(af_ca, crystal_ca)
overall_rmsd = np.sqrt((deviation ** 2).mean())

print(f"Overall CA RMSD over {len(deviation)} matched residues: {overall_rmsd:.2f} Å")

print("\nHighest-deviation residues:")
print(deviation.nlargest(10).round(2).to_string())

print(f"\nDeviation right at the R249S hotspot: {deviation.get(249, float('nan')):.2f} Å")


**What this shows, in a live-verified run:** an overall CA RMSD around 0.7-0.8 Å across
the whole matched domain — AlphaFold's fold-level prediction of the core domain is very
close to the crystal, wild-type-vs-mutant notwithstanding. Deviation right at residue
249 itself is small, similar in size to the average — the R249S mutation changes a side
chain, not the backbone path there. The **highest**-deviation residues instead cluster
immediately **next to the crystal's own unresolved loops** (missing residues printed
above) — which makes sense: a static AlphaFold prediction is compared against one
specific crystal snapshot, and the regions right at the edge of a genuinely
flexible/disordered loop are exactly where a single crystal conformation and a single
predicted conformation are least likely to agree, independent of whether the prediction
is "wrong." ⚠️ Exact numbers depend on the live AlphaFold DB model version — re-verify if
this section is ever re-run and the numbers look different.

In [ ]:
# Locate the residues that actually coordinate the structural Zn (within 2.6 Å, a typical
# direct-coordination distance): confirms the claim above with real geometry, and gives us
# something concrete to zoom in on.
hetatms = atom_records(crystal_text, record="HETATM")
zn_position = hetatms.loc[hetatms["residue_name"] == "ZN", ["x", "y", "z"]].values[0]

zn_coordinating_residues = residues_near_point(crystal_text, zn_position, radius=2.6)
print(f"Residues directly coordinating the structural Zn²⁺: {zn_coordinating_residues}")
print("(A classic Cys/His zinc-finger-like site, present in the crystal, absent from the "
      "AlphaFold model.)")


In [ ]:
# Move the FULL AlphaFold model by the rotation we just measured, so the whole structure
# travels together, then overlay both structures and zoom on what is worth looking at.
af_model_aligned_text = apply_transform(af_model_text, rotation, af_centroid, crystal_centroid)

show_superposition(crystal_text, af_model_aligned_text,
                   highlight=[249] + zn_coordinating_residues, ligand="ZN").show()
print("Blue = 3D08 crystal (R249S mutant, with bound Zn). Orange = AlphaFold's WT model "
      "(no Zn, no ligand).")


---

# End of the guide

One real AlphaFold prediction, taken apart: where the data comes from, what pLDDT and PAE
each answer, what AlphaMissense adds that neither can, how the model compares against the
crystal — and what it leaves out.

**Now open `ex02_workbook.ipynb`.**